# 03 — Aspect models (ABSA)

Seven aspect detectors plus per-aspect polarity, on one shared TF-IDF vectorizer.
Feeds **thesis 4.6** and **5.6**.

This runs before the index, because ASCS needs to know which aspects a review mentions —
and it must read them from *predictions*, not from the annotation. Using annotated aspects
would rely on labels that do not exist when the app is handed a live product URL, and would
inflate everything downstream.


## Kernel check


In [ ]:
import sys; print(sys.executable)


## Train

Fourteen heads, about 8 seconds.


In [ ]:
from IPython.display import Image, display
from trustlens import config, absa
res = absa.report_all()
res['model'].meta


## Detection — does the review mention this aspect?


In [ ]:
res['detection']


## Polarity — how does it feel about it?


In [ ]:
res['polarity']


## The honest view

A macro F1 over three classes hides a lot when one of them has three test examples.
This table gives every class its own row with its support beside it.

**12 of 18 cells have too few test examples to judge.** Only Product Quality has enough
data to support a per-aspect polarity claim; for the rest, only the Positive class is
measurable. Report this table, not the macro column alone.


In [ ]:
absa.polarity_detail(res['model'], res['test'])


In [ ]:
display(Image(filename=str(config.FIGURES/'10_absa_performance.png'), width=620))


## What the model predicts


In [ ]:
m = absa.load()
examples = ['fast delivery but poor quality',
            'ලස්සනයි ඉක්මනට ආවා',
            'too small for me',
            'cheap price and good packaging']
for t, d in zip(examples, m.predict(examples)):
    print(f'{t}\n   -> {d}\n')


## What this means for ASCS in step 4

ASCS counts aspect sentiments that contradict the star direction. To do that it needs
reliable *negative* aspect predictions — and outside Product Quality, the models rarely
produce them, because the training data barely contains them (Delivery/Shipping has 16
negative examples out of 233).

So ASCS will under-detect contradiction, and mostly fire through Product Quality.
That is a real limit, not a bug. The plan already calls for reporting CII twice —
once with predicted aspects and once with annotated ones as an upper bound. The gap
between those two now has a known cause, and quantifying it is itself a result.


## What goes where

| Output | Section |
|---|---|
| `11_absa_detection.csv` | 5.6 — technical measures |
| `12_absa_polarity.csv` | 5.6 |
| `13_absa_polarity_detail.csv` | 5.6, and the limitations discussion |
| figure `10` | 4.6 and 5.6 |
